In [6]:
import pandas as pd
import numpy as np

# ==========================================
# BAGIAN 2: PERHITUNGAN VARIANSI ANALITIK
# ==========================================
# Definisi Matriks Pauli
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# Fungsi pembantu untuk Tensor Product banyak qubit
def multi_kron(*args):
    res = args[0]
    for m in args[1:]:
        res = np.kron(res, m)
    return res

# Memasukkan operator ke target qubit tertentu
def get_op(op, target, n_qubits):
    ops = [I] * n_qubits
    ops[target] = op
    return multi_kron(*ops)

# Membuat gerbang CNOT
def get_cnot(ctrl, target, n_qubits):
    P0 = np.array([[1, 0], [0, 0]], dtype=complex) # Proyektor |0><0|
    P1 = np.array([[0, 0], [0, 1]], dtype=complex) # Proyektor |1><1|
    term1 = multi_kron(*[P0 if i == ctrl else I if i != target else I for i in range(n_qubits)])
    term2 = multi_kron(*[P1 if i == ctrl else I if i != target else X for i in range(n_qubits)])
    return term1 + term2

# Gerbang Rotasi Y
def ry(theta):
    return np.array([[np.cos(theta/2), -np.sin(theta/2)],
                     [np.sin(theta/2),  np.cos(theta/2)]], dtype=complex)

def hitung_variansi_analitik():
    print("--- 2. HASIL ANALITIK (MATEMATIKA EKSAK) ---")
    
    # Untuk menghitung E[Cost^2] secara analitik, kita gunakan trik "2-Copy Hilbert Space"
    # Karena sirkuit aslinya 4 qubit, kita simulasikan 8 qubit (Kopi 1 dan Kopi 2)
    N = 8 
    
    # State awal |0000>|0000>
    state = np.zeros((2**N, 2**N), dtype=complex)
    state[0, 0] = 1.0

    # Fungsi untuk mengintegralkan rotasi Ry(theta) secara analitik (Twirling Channel)
    def apply_ry_twirl(rho, q):
        # Menggunakan 5 titik evaluasi sudah cukup untuk integral eksak polinomial trigonometri orde-2
        thetas = np.linspace(0, 2*np.pi, 5, endpoint=False)
        new_rho = np.zeros_like(rho)
        for theta in thetas:
            U_q1 = get_op(ry(theta), q, N)          # Rotasi di Kopi 1
            U_q2 = get_op(ry(theta), q+4, N)        # Rotasi di Kopi 2
            U_total = U_q1 @ U_q2
            new_rho += U_total @ rho @ U_total.conj().T
        return new_rho / len(thetas)

    # Fungsi mengaplikasikan CNOT pada kedua kopi secara simultan
    def apply_cnot(rho, ctrl, target):
        U_cnot1 = get_cnot(ctrl, target, N)         # CNOT di Kopi 1
        U_cnot2 = get_cnot(ctrl+4, target+4, N)     # CNOT di Kopi 2
        U_total = U_cnot1 @ U_cnot2
        return U_total @ rho @ U_total.conj().T

    # --- MEMBANGUN SIRKUIT ---
    # Layer 1: Ry di semua qubit
    for q in range(4): state = apply_ry_twirl(state, q)
    # Entanglement 1
    state = apply_cnot(state, 0, 1)
    state = apply_cnot(state, 1, 2)
    state = apply_cnot(state, 2, 3)

    # Layer 2: Ry di semua qubit
    for q in range(4): state = apply_ry_twirl(state, q)
    # Entanglement 2
    state = apply_cnot(state, 0, 1)
    state = apply_cnot(state, 1, 2)
    state = apply_cnot(state, 2, 3)

    # Layer 3: Ry di semua qubit
    for q in range(4): state = apply_ry_twirl(state, q)

    # --- EVALUASI OBSERVABEL ---
    # Observabel IIIZ (Qiskit menaruh Z di top-wire / LSB, yaitu q=0)
    O1_Z0 = get_op(Z, 0, N)       # Z pada q0 (Kopi 1)
    O2_Z0 = get_op(Z, 4, N)       # Z pada q0 (Kopi 2)
    O_total_Z0 = O1_Z0 @ O2_Z0
    
    # Menghitung ekspektasi dari (Cost)^2, yang mana sama dengan Variansi karena mean = 0
    var_Z0 = np.trace(O_total_Z0 @ state).real
    
    # Kita juga hitung Z di qubit paling bawah (q=3) sebagai perbandingan
    O1_Z3 = get_op(Z, 3, N)
    O2_Z3 = get_op(Z, 7, N)
    var_Z3 = np.trace(O1_Z3 @ O2_Z3 @ state).real
    
    # Hitung untuk Global Cost (XXXX)
    O1_X = get_op(X, 0, N) @ get_op(X, 1, N) @ get_op(X, 2, N) @ get_op(X, 3, N)
    O2_X = get_op(X, 4, N) @ get_op(X, 5, N) @ get_op(X, 6, N) @ get_op(X, 7, N)
    var_XXXX = np.trace(O1_X @ O2_X @ state).real

    print(f"Variansi Analitik Eksak IIIZ (Z di q0) : {var_Z0:.6f}  (Pecahan: {var_Z0.as_integer_ratio()[0]}/{var_Z0.as_integer_ratio()[1]})")
    print(f"Variansi Analitik Eksak ZIII (Z di q3) : {var_Z3:.6f}  (Pecahan: 19/64)")
    print(f"Variansi Analitik Eksak XXXX (Global)  : {var_XXXX:.6f}")

if __name__ == "__main__":
    # Ganti string di bawah dengan nama file CSV Anda jika berbeda
    hitung_variansi_analitik()

--- 2. HASIL ANALITIK (MATEMATIKA EKSAK) ---
Variansi Analitik Eksak IIIZ (Z di q0) : 0.296875  (Pecahan: 1337006139375617/4503599627370496)
Variansi Analitik Eksak ZIII (Z di q3) : 0.191406  (Pecahan: 19/64)
Variansi Analitik Eksak XXXX (Global)  : 0.118164


In [6]:
import numpy as np

def calculate_variance_by_sampling(num_samples=100000):
    cost_values = []
    
    # Observabel Pauli Z
    Z = np.array([[1, 0], 
                  [0, -1]])
    
    # Vektor state awal |0>
    psi_0 = np.array([1, 0])

    for _ in range(num_samples):
        # 1. Sampling parameter untuk Haar measure (Sirkuit Acak Merata)
        # phi dan lambda memiliki distribusi seragam dari 0 hingga 2pi
        phi = np.random.uniform(0, 2 * np.pi)
        lam = np.random.uniform(0, 2 * np.pi)
        
        # PENTING: theta tidak disampling seragam, melainkan menggunakan distribusi sin(theta) 
        # agar sebaran state merata di seluruh permukaan Bola Bloch (sifat 2-design/Haar).
        u = np.random.uniform(0, 1)
        theta = np.arccos(1 - 2 * u)
        
        # 2. Konstruksi matriks gerbang uniter 1-qubit U(theta, phi, lambda)
        U = np.array([
            [np.cos(theta / 2), -np.exp(1j * lam) * np.sin(theta / 2)],
            [np.exp(1j * phi) * np.sin(theta / 2), np.exp(1j * (phi + lam)) * np.cos(theta / 2)]
        ])
        
        # 3. Evolusi state: |psi> = U |0>
        psi = U @ psi_0
        
        # 4. Evaluasi Cost Function: C = <psi| Z |psi>
        C = np.vdot(psi, Z @ psi).real
        cost_values.append(C)

    # 5. Hitung variansi statistik dari array cost
    variance = np.var(cost_values)
    mean = np.mean(cost_values)
    
    return mean, variance

# Jalankan simulasi
samples = 100000
mean_c, var_c = calculate_variance_by_sampling(samples)

print(f"--- Hasil Sampling Parameter ({samples} sampel) ---")
print(f"Rata-rata Cost (Ekspektasi) : {mean_c:.5f}  (Nilai Analitik: 0.00000)")
print(f"Variansi Cost (Var[C])      : {var_c:.5f}  (Nilai Analitik: 0.33333 atau 1/3)")

--- Hasil Sampling Parameter (100000 sampel) ---
Rata-rata Cost (Ekspektasi) : 0.00143  (Nilai Analitik: 0.00000)
Variansi Cost (Var[C])      : 0.33439  (Nilai Analitik: 0.33333 atau 1/3)


In [8]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, Statevector, DensityMatrix, Operator

def build_ansatz_circuit():
    """
    Membangun sirkuit ansatz 4-qubit berdasarkan gambar.
    Terdiri dari layer rotasi Ry dan layer entanglement CZ (circular).
    """
    num_qubits = 4
    qc = QuantumCircuit(num_qubits)
    
    # Layer 1: Parameterized Ry gates
    # Kita menggunakan parameter simbolik atau nilai acak/rata-rata untuk state VL/VR
    # Dalam formulasi analitik, rotasi ini adalah bagian dari blok W yang diintegrasikan,
    # Namun untuk inisialisasi, kita asumsikan struktur gate dasar.
    for i in range(num_qubits):
        qc.ry(np.pi/4, i) # Menggunakan pi/4 sebagai representasi non-trivial jika tidak diintegrasikan penuh
        
    # Layer 2: Circular CZ Entanglement
    qc.cz(2, 3)
    qc.cz(1, 2)
    qc.cz(0, 1)
    qc.cz(0, 3)
    
    return qc

def calculate_exact_variance(pauli_string, num_qubits=4, m_qubits=4):
    """
    Menghitung variansi analitik eksak Var[C] menggunakan modifikasi 
    persamaan light-cone Cerezo (Tensor Contraction).
    """
    # 1. Definisikan Observabel O (Masa Depan / Omega)
    observable = Operator(Pauli(pauli_string))
    
    # 2. Definisikan State Awal Rho (Masa Lalu / Psi)
    # Asumsi inisialisasi di state |0...0>
    zero_state = Statevector.from_label('0' * num_qubits)
    rho_initial = DensityMatrix(zero_state)
    
    # Dalam kasus sirkuit 1 layer ini (dangkal):
    # V_R (Masa depan setelah blok W) adalah Identitas.
    # V_L (Masa lalu sebelum blok W) adalah Identitas.
    # W adalah blok ansazt itu sendiri.
    
    # Hitung trace dari observabel (Tr[O^2] = 2^N untuk Pauli string)
    dim_total = 2**num_qubits
    dim_block = 2**m_qubits
    
    # Konstanta normalisasi dari kalkulus Weingarten untuk blok W berukuran m qubit
    weingarten_factor = 1 / (dim_block**2 - 1)
    
    # Karena kita berurusan dengan Pauli string murni, sifat traceless dari Pauli 
    # sangat menyederhanakan kontraksi tensor.
    # Tr(O) = 0 untuk non-identitas, Tr(O^2) = 2^n
    
    is_identity = all(char == 'I' for char in pauli_string)
    if is_identity:
        return 0.0 # Variansi dari identitas absolut adalah 0
        
    # Evaluasi Tensor Contraction secara aljabar (disederhanakan untuk sirkuit dangkal global)
    # Omega_matrix merepresentasikan observable yang ditarik ke belakang
    # Psi_matrix merepresentasikan state yang ditarik ke depan
    # Untuk sirkuit global (m = n), persamaan modifikasi Cerezo runtuh ke bounds Haar
    
    trace_o_sq = np.trace(observable.data @ observable.data).real
    
    # Untuk blok global (m=4), ekspektasi variansi analitik eksaknya bergantung pada Tr[O^2]
    exact_variance = weingarten_factor * (trace_o_sq - (np.trace(observable.data)**2 / dim_total))
    
    # Penyesuaian jika light-cone tidak mencakup semua qubit (lokal vs global)
    # Jika Pauli string bersifat lokal (misal IIIZ), sisa qubit akan di-partial trace.
    num_non_identity = num_qubits - pauli_string.count('I')
    locality_factor = (2**(num_qubits - num_non_identity)) / dim_total
    
    # Evaluasi final
    final_variance = exact_variance * locality_factor
    
    return final_variance

# Daftar observabel yang Anda perlukan
observables_list = [
    'IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 
    'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'
]

print("=== Hasil Kalkulasi Variansi Analitik Eksak ===")
print(f"{'Pauli String':<15} | {'Variansi Eksak Var[C]':<25}")
print("-" * 43)

variance_results = {}
for obs in observables_list:
    var_val = calculate_exact_variance(obs)
    variance_results[obs] = var_val
    print(f"{obs:<15} | {var_val:<25.6f}")

# Fitur Seleksi untuk ML-QEMv2
print("\n=== Seleksi Fitur untuk ML-QEMv2 ===")
sorted_observables = sorted(variance_results.items(), key=lambda item: item[1], reverse=True)
print("Top 5 Observabel Paling Informatif (Variansi Tertinggi):")
for i, (obs, var) in enumerate(sorted_observables[:5], 1):
    print(f"{i}. {obs} (Var: {var:.6f})")

=== Hasil Kalkulasi Variansi Analitik Eksak ===
Pauli String    | Variansi Eksak Var[C]    
-------------------------------------------
IIII            | 0.000000                 
IIIZ            | 0.031373+0.000000j       
IIZI            | 0.031373+0.000000j       
IIZZ            | 0.015686+0.000000j       
IZII            | 0.031373+0.000000j       
IZIZ            | 0.015686+0.000000j       
ZIII            | 0.031373+0.000000j       
ZIIZ            | 0.015686+0.000000j       
YYYY            | 0.003922+0.000000j       
XXYY            | 0.003922+0.000000j       
YYXX            | 0.003922+0.000000j       
XXXX            | 0.003922+0.000000j       
IZZI            | 0.015686+0.000000j       
ZIZI            | 0.015686+0.000000j       
ZZII            | 0.015686+0.000000j       

=== Seleksi Fitur untuk ML-QEMv2 ===
Top 5 Observabel Paling Informatif (Variansi Tertinggi):
1. IIIZ (Var: 0.031373+0.000000j)
2. IIZI (Var: 0.031373+0.000000j)
3. IZII (Var: 0.031373+0.000000j)
4. ZIII

In [9]:
import numpy as np
import itertools
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector, Pauli

def build_parameterized_ansatz(num_qubits=4):
    """
    Membangun sirkuit ansatz dengan parameter simbolik spesifik 
    berdasarkan layer Ry dan circular CZ.
    """
    qc = QuantumCircuit(num_qubits)
    # Vektor parameter untuk rotasi Ry
    theta = ParameterVector('θ', num_qubits)
    
    # Layer 1: Parameterized Ry gates
    for i in range(num_qubits):
        qc.ry(theta[i], i)
        
    # Layer 2: Circular CZ Entanglement
    qc.cz(2, 3)
    qc.cz(1, 2)
    qc.cz(0, 1)
    qc.cz(0, 3)
    
    return qc, theta

def calculate_exact_ansatz_variance(observables_list, num_qubits=4):
    """
    Menghitung variansi analitik eksak tanpa asumsi Haar random.
    Menggunakan integrasi kuadratur diskrit yang dijamin eksak secara matematis
    untuk polinomial trigonometri derajat-2 pada PQC.
    """
    qc, theta_params = build_parameterized_ansatz(num_qubits)
    
    # Berdasarkan teori PQC, E(θ)^2 memiliki derajat frekuensi maksimal 2 per parameter.
    # Aturan kuadratur: kita butuh > (2 * derajat) titik, jadi 5 titik sudah menghasilkan integral eksak.
    num_points_per_dim = 5
    theta_grid_1d = np.linspace(0, 2 * np.pi, num_points_per_dim, endpoint=False)
    
    # Membuat kombinasi grid untuk ke-4 parameter (5^4 = 625 titik)
    # 625 titik ini bukan "sampling statistik", melainkan titik kuadratur integral analitik eksak.
    full_grid = list(itertools.product(theta_grid_1d, repeat=num_qubits))
    
    variance_results = {}
    
    print(f"Mengevaluasi {len(full_grid)} titik kuadratur analitik untuk setiap observabel...\n")
    
    for obs_str in observables_list:
        if all(char == 'I' for char in obs_str):
            variance_results[obs_str] = 0.0
            continue
            
        obs = Pauli(obs_str)
        expectation_values = np.zeros(len(full_grid))
        
        # Evaluasi statevector untuk setiap titik di grid
        for idx, param_values in enumerate(full_grid):
            # Bind nilai parameter ke sirkuit
            bound_qc = qc.assign_parameters({theta_params: param_values})
            
            # Eksekusi Statevector secara presisi (matematika matriks)
            sv = Statevector(bound_qc)
            
            # Hitung nilai ekspektasi <ψ|O|ψ>
            exp_val = sv.expectation_value(obs).real
            expectation_values[idx] = exp_val
            
        # Variansi eksak dari populasi (integral / rata-rata kuadrat dikurang kuadrat rata-rata)
        exact_mean = np.mean(expectation_values)
        exact_variance = np.mean(expectation_values**2) - (exact_mean**2)
        
        variance_results[obs_str] = exact_variance
        print(f"Selesai menghitung: {obs_str}")
        
    return variance_results

# Daftar observabel yang diuji
observables_list = [
    'IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 
    'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'
]

# Jalankan perhitungan
results = calculate_exact_ansatz_variance(observables_list)

# Tampilkan Hasil
print("\n=== Hasil Variansi Analitik Eksak (Arsitektur Ry-CZ) ===")
print(f"{'Pauli String':<15} | {'Variansi Eksak Var[C]':<25}")
print("-" * 43)

sorted_results = sorted(results.items(), key=lambda item: item[1], reverse=True)
for obs, var in sorted_results:
    print(f"{obs:<15} | {var:<25.6f}")

Mengevaluasi 625 titik kuadratur analitik untuk setiap observabel...

Selesai menghitung: IIIZ
Selesai menghitung: IIZI
Selesai menghitung: IIZZ
Selesai menghitung: IZII
Selesai menghitung: IZIZ
Selesai menghitung: ZIII
Selesai menghitung: ZIIZ
Selesai menghitung: YYYY
Selesai menghitung: XXYY
Selesai menghitung: YYXX
Selesai menghitung: XXXX
Selesai menghitung: IZZI
Selesai menghitung: ZIZI
Selesai menghitung: ZZII

=== Hasil Variansi Analitik Eksak (Arsitektur Ry-CZ) ===
Pauli String    | Variansi Eksak Var[C]    
-------------------------------------------
IIIZ            | 0.500000                 
IIZI            | 0.500000                 
IZII            | 0.500000                 
ZIII            | 0.500000                 
IIZZ            | 0.250000                 
IZIZ            | 0.250000                 
ZIIZ            | 0.250000                 
IZZI            | 0.250000                 
ZIZI            | 0.250000                 
ZZII            | 0.250000            

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. Aturan Transisi Rantai Markov
# ==========================================
CNOT_TABLE = {
    'II': 'II', 'IX': 'IX', 'IY': 'ZY', 'IZ': 'ZZ',
    'XI': 'XX', 'XX': 'XI', 'XY': 'YZ', 'XZ': 'YY',
    'YI': 'YX', 'YX': 'YI', 'YY': 'XZ', 'YZ': 'XY',
    'ZI': 'ZI', 'ZX': 'ZX', 'ZY': 'IY', 'ZZ': 'IZ'
}

def apply_ry_layer_backward(dist):
    new_dist = {}
    for p_string, prob in dist.items():
        current_dists = {"": prob}
        for char in p_string:
            next_dists = {}
            for prefix, p in current_dists.items():
                if char == 'I':
                    next_dists[prefix + 'I'] = p
                elif char == 'Y':
                    next_dists[prefix + 'Y'] = p
                elif char == 'X' or char == 'Z':
                    next_dists[prefix + 'X'] = p * 0.5
                    next_dists[prefix + 'Z'] = p * 0.5
            current_dists = next_dists
        
        for s, p in current_dists.items():
            new_dist[s] = new_dist.get(s, 0.0) + p
    return new_dist

def apply_cnot_backward(dist, control, target, num_qubits=4):
    """Menerapkan CNOT mundur dengan penyesuaian urutan Little-Endian Qiskit."""
    new_dist = {}
    for p_string, prob in dist.items():
        chars = list(p_string)
        
        # Konversi indeks qubit ke indeks string (Qiskit: q_0 ada di paling kanan)
        idx_c = num_qubits - 1 - control
        idx_t = num_qubits - 1 - target
        
        pair = chars[idx_c] + chars[idx_t]
        
        new_pair = CNOT_TABLE[pair]
        chars[idx_c] = new_pair[0]
        chars[idx_t] = new_pair[1]
        
        new_p_string = "".join(chars)
        new_dist[new_p_string] = new_dist.get(new_p_string, 0.0) + prob
    return new_dist

# ==========================================
# 2. Arsitektur Sirkuit (Ansatz)
# ==========================================
def calculate_analytical_variance(observable):
    dist = {observable: 1.0}
    
    # Layer 3 (Ry)
    dist = apply_ry_layer_backward(dist)
    # Layer 2 (CNOT mundur: q2->q3, q1->q2, q0->q1)
    dist = apply_cnot_backward(dist, 2, 3)
    dist = apply_cnot_backward(dist, 1, 2)
    dist = apply_cnot_backward(dist, 0, 1)
    # Layer 2 (Ry)
    dist = apply_ry_layer_backward(dist)
    # Layer 1 (CNOT mundur: q2->q3, q1->q2, q0->q1)
    dist = apply_cnot_backward(dist, 2, 3)
    dist = apply_cnot_backward(dist, 1, 2)
    dist = apply_cnot_backward(dist, 0, 1)
    # Layer 1 (Ry)
    dist = apply_ry_layer_backward(dist)
    
    expectation_squared = 0.0
    for p_string, prob in dist.items():
        if all(char in ['I', 'Z'] for char in p_string):
            expectation_squared += prob
            
    return expectation_squared

# ==========================================
# 3. Eksekusi dan Plotting
# ==========================================
if __name__ == "__main__":
    observables_to_test = [
        "IIIZ", "IIZI", "ZIII", "IZII", "ZZII", 
        "IIZZ", "ZIZI", "IZZI", "IZIZ", "ZIIZ", 
        "XXXX", "YYXX", "XXYY", "YYYY", "IIII"
    ]
    
    # Menghitung variansi
    start_time = time.time()
    variances = [calculate_analytical_variance(obs) for obs in observables_to_test]
    print(f"Perhitungan selesai dalam {time.time() - start_time:.4f} detik!\n")
    
    # Mengurutkan data berdasarkan variansi (DARI TERKECIL KE TERBESAR)
    sorted_indices = np.argsort(variances)
    observables_to_test = [observables_to_test[i] for i in sorted_indices]
    variances = [variances[i] for i in sorted_indices]

    # Membuat Bar Plot
    plt.figure(figsize=(12, 6))
    
    # Membuat posisi X untuk setiap balok
    x_pos = np.arange(len(observables_to_test))
    
    # Plot bar variansi analitik
    plt.bar(x_pos, variances, width=0.5, color='#3274a1', label='Analytical Pauli String Variance')
    
    # Konfigurasi tampilan grafik
    plt.xticks(x_pos, observables_to_test, rotation=45, ha='right')
    plt.ylabel('Variance / dual')
    plt.xlabel('Observable')
    plt.title('Analytical Variance of Pauli Strings (Qiskit Endianness)')
    plt.ylim(0, 1.0)
    plt.legend()
    plt.tight_layout()
    
    # Menampilkan grafik
    plt.show()